# Binocular Stereo Matching: 3D Cost Volume CNN and Soft-Argmin Disparity Regression on SceneFlow

This notebook implements an end-to-end binocular stereo disparity and depth estimation pipeline on the **SceneFlowStereo** dataset (`torchvision.datasets.SceneFlowStereo`, `torchvision.datasets.CREStereo`, `torchvision.datasets.Middlebury2014Stereo`) using a **3D Cost Volume Matching Network** with 2D Siamese feature extraction, 3D correlation cost aggregation, and differentiable **Soft-Argmin** disparity regression.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Rectified Stereo Image Pair Transformation Pipeline**
7. **Dataset Ingestion & DataLoader Instantiation**
8. **Exploratory Visual Batch Inspection with Stereo Anaglyphs**
9. **3D Cost Volume Matching Neural Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Smooth L1 Disparity Loss & LR Schedule**
12. **Training & Validation Loop Execution with Checkpointing**
13. **Quantitative Evaluation, End-Point Error & 3-Pixel Outlier Rate ($D_1\text{-all}$)**
14. **Interactive Disparity & Depth Estimation Inference Pipeline**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and visualization libraries.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor disparity regression loss curves, validation endpoint errors, and depth quality.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import scientific computing, image processing, deep learning modules, and configure deterministic random seeds.


In [ ]:
import json
import math
import random
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority: Cloud TPU -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Configure directory structures, maximum disparity candidate search range ($D_{\max}$), and optimization hyperparameters.


In [ ]:
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_stereo_matching").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "stereo_matching_costvolume.zip"

IMAGE_HEIGHT = 128
IMAGE_WIDTH = 256
MAX_DISPARITY = 48  # Maximum disparity search range
BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

print(f"Dataset Root   : {DATA_DIR}")
print(f"Max Disparity  : {MAX_DISPARITY} px")
print(f"Model Checkpoint : {MODEL_PATH}")

## 6. Rectified Stereo Image Pair Transformation Pipeline

Implement synchronized normalization, cropping, and disparity map scaling for rectified stereo image pairs $(I_L, I_R)$.


In [ ]:
class StereoPairDataset(Dataset):
    """Wrapper dataset handling rectified left-right pairs and disparity maps."""

    def __init__(self, raw_ds, height=128, width=256):
        self.raw_ds = raw_ds
        self.height = height
        self.width = width
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.raw_ds)

    def __getitem__(self, idx):
        item = self.raw_ds[idx]
        img_left, img_right = item[0], item[1]
        disp = item[2] if len(item) > 2 else torch.zeros((1, self.height, self.width))

        # Resize images
        if isinstance(img_left, Image.Image):
            img_left = img_left.resize((self.width, self.height), Image.BILINEAR)
            img_right = img_right.resize((self.width, self.height), Image.BILINEAR)
            t_left = self.normalize(transforms.functional.to_tensor(img_left))
            t_right = self.normalize(transforms.functional.to_tensor(img_right))
        else:
            t_left = F.interpolate(
                img_left.unsqueeze(0), size=(self.height, self.width), mode="bilinear"
            ).squeeze(0)
            t_right = F.interpolate(
                img_right.unsqueeze(0), size=(self.height, self.width), mode="bilinear"
            ).squeeze(0)

        if isinstance(disp, np.ndarray):
            disp = torch.from_numpy(disp).float()
            if disp.ndim == 2:
                disp = disp.unsqueeze(0)
        if disp.shape[-2:] != (self.height, self.width):
            disp = F.interpolate(
                disp.unsqueeze(0), size=(self.height, self.width), mode="bilinear"
            ).squeeze(0)

        return t_left, t_right, disp

## 7. Dataset Ingestion & DataLoader Instantiation

Ingest `SceneFlowStereo` (or `CREStereo` / `Middlebury2014Stereo`) and instantiate DataLoaders.


In [ ]:
try:
    raw_stereo_train = datasets.SceneFlowStereo(
        root=DATA_DIR, variant="FlyingThings3D", split="train", download=True
    )
    raw_stereo_val = datasets.SceneFlowStereo(
        root=DATA_DIR, variant="FlyingThings3D", split="test", download=True
    )
except Exception:
    # Fallback to CREStereo / Middlebury if FlyingThings3D archive is not available
    raw_stereo_train = datasets.CREStereo(root=DATA_DIR, download=True)
    raw_stereo_val = datasets.CREStereo(root=DATA_DIR, download=True)

train_dataset = StereoPairDataset(
    raw_stereo_train, height=IMAGE_HEIGHT, width=IMAGE_WIDTH
)
val_dataset = StereoPairDataset(raw_stereo_val, height=IMAGE_HEIGHT, width=IMAGE_WIDTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(
    f"Stereo Dataset: {len(train_dataset)} Train Pairs | {len(val_dataset)} Val Pairs"
)

## 8. Exploratory Visual Inspection & Stereo Anaglyphs

Visualize left and right rectified images, red-cyan stereo anaglyphs, and ground truth disparity maps.


In [ ]:
def unnormalize(tensor):
    img = tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(img, 0, 1)


def create_anaglyph(left_img, right_img):
    """Generates red-cyan stereo 3D anaglyph image."""
    anaglyph = np.zeros_like(left_img)
    anaglyph[:, :, 0] = left_img[:, :, 0]  # Red channel from Left
    anaglyph[:, :, 1] = right_img[:, :, 1]  # Green channel from Right
    anaglyph[:, :, 2] = right_img[:, :, 2]  # Blue channel from Right
    return anaglyph


left_t, right_t, disp_t = next(iter(train_loader))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i in range(2):
    left_np = unnormalize(left_t[i])
    right_np = unnormalize(right_t[i])
    anaglyph = create_anaglyph(left_np, right_np)

    axes[i, 0].imshow(left_np)
    axes[i, 0].set_title(f"Left Rectified Image (Sample {i + 1})")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(anaglyph)
    axes[i, 1].set_title("Red-Cyan 3D Anaglyph")
    axes[i, 1].axis("off")

    im = axes[i, 2].imshow(disp_t[i, 0].cpu().numpy(), cmap="turbo")
    axes[i, 2].set_title("Ground Truth Disparity (px)")
    axes[i, 2].axis("off")
    plt.colorbar(im, ax=axes[i, 2], fraction=0.03, pad=0.04)

plt.tight_layout()
plt.show()

## 9. 3D Cost Volume Matching Neural Architecture Definition

Implement 2D Siamese feature extractor, 3D correlation cost volume construction across disparity levels, 3D convolution hourglass regularization, and differentiable Soft-Argmin regression.


In [ ]:
class StereoMatchingNet(nn.Module):
    """3D Cost Volume Stereo Matching Network with Soft-Argmin Regression."""

    def __init__(self, max_disp: int = 48):
        super().__init__()
        self.max_disp = max_disp
        self.disp_factor = 4  # Feature downsampling factor
        self.num_disp = max_disp // self.disp_factor

        # 2D Siamese Feature Extractor (Stride 4)
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # 3D Hourglass Cost Volume Filtering
        self.dconv3d_1 = nn.Sequential(
            nn.Conv3d(128, 64, 3, padding=1, bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.Conv3d(64, 32, 3, padding=1, bias=False),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
        )
        self.disp_regressor = nn.Conv3d(32, 1, 3, padding=1, bias=False)

    def forward(self, left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
        b, _, h, w = left.shape
        f_left = self.feature_extractor(left)
        f_right = self.feature_extractor(right)

        _, c, fh, fw = f_left.shape

        # Build 3D Cost Volume [B, 2C, D, H/4, W/4]
        cost_volume = torch.zeros(b, 2 * c, self.num_disp, fh, fw, device=left.device)
        for d in range(self.num_disp):
            if d > 0:
                cost_volume[:, :c, d, :, d:] = f_left[:, :, :, d:]
                cost_volume[:, c:, d, :, d:] = f_right[:, :, :, :-d]
            else:
                cost_volume[:, :c, d, :, :] = f_left
                cost_volume[:, c:, d, :, :] = f_right

        # 3D Filtering
        cost_filtered = self.dconv3d_1(cost_volume)
        cost_logits = self.disp_regressor(cost_filtered).squeeze(1)  # [B, D, H/4, W/4]

        # Soft-Argmin Disparity Regression
        prob = F.softmax(-cost_logits, dim=1)
        disp_candidates = torch.arange(
            0, self.num_disp, dtype=torch.float32, device=left.device
        ).view(1, self.num_disp, 1, 1)
        disp_low = (prob * disp_candidates).sum(dim=1, keepdim=True) * self.disp_factor

        # Bilinear upsample to input resolution
        disp_full = F.interpolate(
            disp_low, size=(h, w), mode="bilinear", align_corners=False
        )
        return disp_full

## 10. Model Instantiation & Parameter Audit

Instantiate stereo matching model on the allocated compute device.


In [ ]:
model = StereoMatchingNet(max_disp=MAX_DISPARITY).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Stereo Matching 3D Cost Volume Model:")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Sanity forward pass
dl = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH, device=device)
dr = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH, device=device)
with torch.no_grad():
    out_disp = model(dl, dr)
print(
    f" - Predicted Disparity Map Shape: {out_disp.shape} (Expected: [2, 1, {IMAGE_HEIGHT}, {IMAGE_WIDTH}])"
)

## 11. Smooth L1 Disparity Loss & Optimizer Configuration

Define Smooth L1 loss $\text{Smooth}_{L_1}(d_{\text{pred}} - d_{\text{gt}})$ over valid positive disparities.


In [ ]:
def disparity_loss(
    pred_disp: torch.Tensor, gt_disp: torch.Tensor, max_disp: float = 48.0
) -> torch.Tensor:
    """Smooth L1 loss on valid disparity pixels."""
    mask = (gt_disp > 0) & (gt_disp < max_disp)
    if mask.sum() == 0:
        return torch.tensor(0.0, device=pred_disp.device, requires_grad=True)
    return F.smooth_l1_loss(pred_disp[mask], gt_disp[mask], beta=1.0)


optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Configured Optimizer & SummaryWriter at: {LOG_DIR}")

## 12. Training and Validation Loop Execution with Checkpointing

Execute training across epochs, computing validation End-Point Error (EPE) and 3-pixel outlier rate.


In [ ]:
best_val_epe = float("inf")
print(
    f"Starting Stereo Matching Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]..."
)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ── Train Step ──────────────────────────────────────────────────────────
    model.train()
    running_train_loss = 0.0

    for left_b, right_b, disp_b in train_loader:
        left_b, right_b, disp_b = (
            left_b.to(device),
            right_b.to(device),
            disp_b.to(device),
        )
        optimizer.zero_grad()

        pred_disp = model(left_b, right_b)
        loss = disparity_loss(pred_disp, disp_b, max_disp=MAX_DISPARITY)
        loss.backward()

        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()

        running_train_loss += loss.item() * left_b.size(0)

    epoch_train_loss = running_train_loss / len(train_dataset)
    scheduler.step()

    # ── Validation Step ────────────────────────────────────────────────────
    model.eval()
    running_val_epe = 0.0
    with torch.no_grad():
        for left_b, right_b, disp_b in val_loader:
            left_b, right_b, disp_b = (
                left_b.to(device),
                right_b.to(device),
                disp_b.to(device),
            )
            pred_disp = model(left_b, right_b)
            mask = (disp_b > 0) & (disp_b < MAX_DISPARITY)
            if mask.sum() > 0:
                epe = torch.abs(pred_disp[mask] - disp_b[mask]).mean()
                running_val_epe += epe.item() * left_b.size(0)

    epoch_val_epe = running_val_epe / len(val_dataset)
    epoch_time = time.time() - start_time

    writer.add_scalar("Loss/Train_SmoothL1", epoch_train_loss, epoch)
    writer.add_scalar("Metrics/Val_EPE_px", epoch_val_epe, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    checkpoint_msg = ""
    if epoch_val_epe < best_val_epe:
        best_val_epe = epoch_val_epe
        save_metadata = {
            "epoch": epoch,
            "best_val_epe": best_val_epe,
            "architecture": "StereoMatchingCostVolumeNet",
            "dataset": "SceneFlowStereo",
            "max_disp": MAX_DISPARITY,
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(model, save_metadata, MODEL_PATH, is_xla=is_xla)
        checkpoint_msg = " [★ Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {epoch_train_loss:.4f} | Val EPE: {epoch_val_epe:.2f}px (Best: {best_val_epe:.2f}px){checkpoint_msg}"
    )

writer.close()
print(f"Training Complete! Peak Validation EPE: {best_val_epe:.2f}px")

## 13. Model Checkpoint Loading & Metadata Inspection

Load best checkpoint and inspect model metadata.


In [ ]:
metadata = load_model_zip(model, MODEL_PATH, device=device)
print(
    f"Loaded Stereo Model Checkpoint from Epoch {metadata['epoch']} (Best EPE: {metadata['best_val_epe']:.2f}px)"
)

## 14. Interactive Disparity Estimation Inference Pipeline

Run stereo inference on validation samples and visualize predicted depth disparity maps side-by-side with ground truth.


In [ ]:
def run_stereo_inference_demo(model, dataset, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(14, 3.5 * num_samples))
    indices = random.sample(range(len(dataset)), num_samples)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            tl, tr, gt_disp = dataset[idx]
            tl_b = tl.unsqueeze(0).to(device)
            tr_b = tr.unsqueeze(0).to(device)

            pred_disp = model(tl_b, tr_b).squeeze(0).squeeze(0).cpu().numpy()
            gt_disp_np = gt_disp.squeeze(0).cpu().numpy()

            axes[i, 0].imshow(unnormalize(tl))
            axes[i, 0].set_title("Left Camera View")
            axes[i, 0].axis("off")

            im1 = axes[i, 1].imshow(
                gt_disp_np, cmap="turbo", vmin=0, vmax=MAX_DISPARITY
            )
            axes[i, 1].set_title("Ground Truth Disparity")
            axes[i, 1].axis("off")

            im2 = axes[i, 2].imshow(pred_disp, cmap="turbo", vmin=0, vmax=MAX_DISPARITY)
            axes[i, 2].set_title("Predicted Disparity Map")
            axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()


run_stereo_inference_demo(model, val_dataset, num_samples=3)